In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

print("OpenRouter API key loaded successfully.")

OpenRouter API key loaded successfully.


In [2]:
from langchain_openrouter import ChatOpenRouter

model = ChatOpenRouter(
    model="z-ai/glm-5.3-flash",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0
)

print("Model initialized successfully")

Model initialized successfully


In [3]:
response = model.invoke(
    "Explain hypertension in simple words."
)

print(response.content)

# Hypertension (High Blood Pressure)

**The basic idea:** Your heart pumps blood through your blood vessels, and the blood pushes against the walls of those vessels. That pushing force is your *blood pressure*. Hypertension just means that force is **too high, too often**.

**A simple analogy:** Think of your blood vessels like garden hoses. Water needs some pressure to flow — that's normal. But if the pressure stays too high for too long, it wears out the hose. The same thing happens to your arteries and organs.

**Why it's a problem:** Over years, high pressure damages blood vessels and strains organs, raising the risk of:
- Heart attacks
- Strokes
- Kidney disease
- Vision problems

**The tricky part:** Hypertension usually has **no symptoms**. You can feel perfectly fine while it silently causes damage. That's why doctors call it the "silent killer" and check your blood pressure at every visit.

**What raises the risk:**
- Getting older, family history
- Too much salt in the diet
-

In [4]:
import requests
import xml.etree.ElementTree as ET

from langchain_core.tools import tool


@tool
def medical_information(topic: str) -> str:
    """
    Search MedlinePlus for general medical information about a topic.
    Provides educational information only and does not diagnose or prescribe.
    """

    url = "https://wsearch.nlm.nih.gov/ws/query"

    params = {
        "db": "healthTopics",
        "term": topic,
        "retmax": 3,
        "rettype": "brief"
    }

    try:
        response = requests.get(
            url,
            params=params,
            timeout=10
        )

        response.raise_for_status()

        root = ET.fromstring(response.text)

        results = []

        for document in root.findall(".//document"):

            title = ""
            summary = ""
            page_url = document.attrib.get("url", "")

            for content in document.findall("content"):

                name = content.attrib.get("name")
                text = "".join(content.itertext()).strip()

                if name == "title":
                    title = text

                elif name == "full-summary":
                    summary = text

            if title or summary:
                results.append(
                    {
                        "title": title,
                        "summary": summary,
                        "url": page_url
                    }
                )

        if not results:
            return f"No medical information found for: {topic}"

        output = f"Medical information from MedlinePlus for '{topic}':\n\n"

        for i, result in enumerate(results, 1):

            output += f"{i}. {result['title']}\n"
            output += f"{result['summary']}\n"
            output += f"Source: {result['url']}\n\n"

        output += (
            "Important: This information is for educational purposes only. "
            "It does not provide a diagnosis or medical prescription."
        )

        return output

    except requests.RequestException as e:
        return f"Unable to access MedlinePlus: {str(e)}"

    except ET.ParseError:
        return "Unable to process the medical information returned by MedlinePlus."

In [5]:
model_with_tools = model.bind_tools(
    [medical_information]
)

In [6]:
response = model_with_tools.invoke(
    "Can you tell me about hypertension?"
)

print(response.tool_calls)

[{'name': 'medical_information', 'args': {'topic': 'hypertension'}, 'id': 'call_7dc6a3702fc940d3a2294ad2', 'type': 'tool_call'}]


In [7]:
! pip install -U faster-whisper sounddevice scipy

  Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl.metadata (60 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 18.2 MB/s  0:00:00
   ---------------------------------------- 0.0/19.2 MB ? eta -:--:--
   --------------- ------------------------ 7.6/19.2 MB 36.2 MB/s eta 0:00:01
   ---------------------- ----------------- 10.7/19.2 MB 37.2 MB/s eta 0:00:01
   ------------------------------- -------- 14.9/19.2 MB 23.5 MB/s eta 0:00:01
   ---------------------------------------- 19.2/19.2 MB 24.2 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 15.9 MB/s  0:00:00
Using cached scipy-1.17.1-cp311-cp311-win_amd64.whl (36.6 MB)
   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
   --------- ------------------------------ 6.8/27.6 MB 32.3 MB/s eta 0:00:01
   -------------------- ------------------- 13.9/27

In [8]:
import sounddevice as sd
from scipy.io.wavfile import write
from faster_whisper import WhisperModel

c:\Users\Sudhanshu Singh\OneDrive\Documents\Agentic-AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
whisper_model = WhisperModel(
    "small",
    device="cpu",
    compute_type="int8"
)

print("Whisper model loaded")

c:\Users\Sudhanshu Singh\OneDrive\Documents\Agentic-AI\venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Sudhanshu Singh\.cache\huggingface\hub\models--Systran--faster-whisper-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Whisper model loaded


In [13]:
! pip install -U edge_tts

In [15]:
import edge_tts

print("edge-tts imported successfully")

edge-tts imported successfully


In [16]:
import edge_tts

communicate = edge_tts.Communicate(
    text="Hello, this is a test of the medical assistant voice.",
    voice="en-IN-NeerjaNeural"
)

await communicate.save("test_voice.mp3")

print("Voice file created successfully")

Voice file created successfully


In [17]:
from IPython.display import Audio, display

display(Audio("test_voice.mp3"))

In [23]:


import asyncio
import edge_tts

from IPython.display import Audio, display
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage



user_text = input("🧑 Patient: ")

print("\nPatient said:")
print(user_text)




system_prompt = """
You are a medical information voice assistant.

Your job is to provide general educational medical information.

Important rules:
- Do not diagnose diseases.
- Do not prescribe medicines.
- Do not replace a healthcare professional.
- Use information retrieved from the medical information tool.
- Since your response will be spoken aloud, keep the answer concise.
- Use simple language.
- Do not use Markdown.
- Avoid long lists.
"""



messages = [
    SystemMessage(
        content=system_prompt
    ),
    HumanMessage(
        content=user_text
    )
]

response = model_with_tools.invoke(messages)

if response.tool_calls:

    messages.append(response)

    print("\nTool Calling...")

    for tool_call in response.tool_calls:

        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        print("Tool:", tool_name)
        print("Arguments:", tool_args)

        if tool_name == "medical_information":

            tool_result = medical_information.invoke(tool_args)

            print("\nTool result:")
            print(tool_result)

            messages.append(
                ToolMessage(
                    content=str(tool_result),
                    tool_call_id=tool_call["id"]
                )
            )

    # Ask the model to produce the final answer
    final_response = model.invoke(messages)

    print("\nFinal response:")
    print(final_response.content)

    answer = final_response.content

else:

    answer = response.content

print("\nMedical Assistant:")
print(answer)


async def generate_voice(text):

    voice = "en-IN-NeerjaNeural"

    communicate = edge_tts.Communicate(
        text=text,
        voice=voice
    )

    await communicate.save(
        "medical_response.mp3"
    )


await generate_voice(answer)

print("\nMedical Assistant Voice:")

display(
    Audio(
        "medical_response.mp3"
    )
)


Patient said:
sickness symptoms

Tool Calling...
Tool: medical_information
Arguments: {'topic': 'common symptoms of illness'}

Tool result:
Medical information from MedlinePlus for 'common symptoms of illness':

1. Foodborne <span class="qt4">Illness</span>

Source: https://medlineplus.gov/foodborneillness.html

2. Reye Syndrome

Source: https://medlineplus.gov/reyesyndrome.html

3. Myalgic Encephalomyelitis/Chronic Fatigue Syndrome

Source: https://medlineplus.gov/myalgicencephalomyelitischronicfatiguesyndrome.html

Important: This information is for educational purposes only. It does not provide a diagnosis or medical prescription.

Final response:
When people get sick, some common symptoms include fever, tiredness, coughing, sore throat, body aches, headache, nausea, and diarrhea. The exact symptoms depend on what kind of illness it is, like a cold, the flu, or a stomach bug.

Can you tell me more about the type of sickness you're curious about, or the symptoms you're experiencing?